In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU, SimpleRNN, Dropout, Input

In [28]:
yahoo_df = pd.read_csv("yahoo_finance_historical.csv")
news_df = pd.read_csv("rss_news_sentiment.csv")
av_df = pd.read_csv("av_timeseries.csv")

In [29]:

# 1.1 Map Sentiment text
sentiment_map = {'Positive': 1, 'Neutral': 0, 'Negative': -1}
news_df['Sentiment'] = news_df['Sentiment'].map(sentiment_map)

# 1.2 FIX: Force Direction to be numeric (In case it has "Up"/"Down" strings)
if av_df['Direction'].dtype == 'object':
    direction_map = {'Up': 1, 'Down': 0, '1': 1, '0': 0}
    av_df['Direction'] = av_df['Direction'].map(direction_map)

# 1.3 Fix Timezones
news_df['Date'] = pd.to_datetime(news_df['Date']).dt.tz_localize(None)
av_df['Date'] = pd.to_datetime(av_df['Date']).dt.tz_localize(None)

# 1.4 Aggregate and Merge
daily_sentiment = news_df.groupby('Date')['Sentiment'].mean().reset_index()
df = pd.merge(av_df, daily_sentiment, on='Date', how='left')

# 1.5 FINAL DATA CHECK: Drop rows with NaN and ensure all are float/int
df['Sentiment'] = df['Sentiment'].fillna(0)
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'Volatility', 'Sentiment']
target = 'Direction'

# Drop any row where target is still NaN after merge
df = df.dropna(subset=[target])

# Convert everything to float32 to satisfy Keras
df[features] = df[features].apply(pd.to_numeric, errors='coerce').astype('float32')
df[target] = pd.to_numeric(df[target], errors='coerce').astype('float32')

df = df.sort_values('Date').set_index('Date')

In [30]:

scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(df[features])

def create_sequences(data, target_data, window_size=10):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:(i + window_size)])
        y.append(target_data.iloc[i + window_size])
    return np.array(X, dtype='float32'), np.array(y, dtype='float32')

WINDOW_SIZE = 10
X, y = create_sequences(scaled_features, df[target], WINDOW_SIZE)

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [31]:

def build_sequential_model(model_type='LSTM'):
    model = Sequential()
    # Using Input layer to fix UserWarning
    model.add(Input(shape=(WINDOW_SIZE, len(features))))
    
    if model_type == 'RNN':
        model.add(SimpleRNN(64, return_sequences=False))
    elif model_type == 'LSTM':
        model.add(LSTM(64, return_sequences=False))
    elif model_type == 'GRU':
        model.add(GRU(64, return_sequences=False))
    
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [32]:


model_names = ['RNN', 'LSTM', 'GRU']
performance = {}

for name in model_names:
    print(f"\n--- Training {name} Model ---")
    model = build_sequential_model(name)
    # Ensure y_train is reshaped correctly for the loss function
    model.fit(X_train, y_train, epochs=15, batch_size=16, verbose=1, validation_split=0.1)
    
    y_probs = model.predict(X_test)
    y_preds = (y_probs > 0.5).astype(int)
    
    performance[name] = {
        'Accuracy': accuracy_score(y_test, y_preds),
        'F1-Score': f1_score(y_test, y_preds),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_probs))
    }

print("\nFinal Model Comparison Results:")
print(pd.DataFrame(performance).T)


--- Training RNN Model ---
Epoch 1/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 139ms/step - accuracy: 0.3750 - loss: 0.7469 - val_accuracy: 0.1250 - val_loss: 0.7394
Epoch 2/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.4531 - loss: 0.6956 - val_accuracy: 0.2500 - val_loss: 0.7394
Epoch 3/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.5000 - loss: 0.7053 - val_accuracy: 0.1250 - val_loss: 0.7430
Epoch 4/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.5625 - loss: 0.6840 - val_accuracy: 0.1250 - val_loss: 0.7436
Epoch 5/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.4062 - loss: 0.7061 - val_accuracy: 0.1250 - val_loss: 0.7434
Epoch 6/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.6094 - loss: 0.6777 - val_accuracy: 0.2500 - val_loss: 0.7464
Epoch 7/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.5625 - loss: 0.6839 - val_accuracy: 0.2500 - val_loss: 0.7518
Epoch 8/15
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.6406 - loss: 0.6744 - val_accurac

In [36]:
import mlflow
import mlflow.keras
mlflow.set_tracking_uri("http://127.0.0.1:5000") 

# 1. Setup the Experiment
mlflow.set_experiment("RealTime_Market_Prediction")

model_names = ['RNN', 'LSTM', 'GRU']
performance = {}

for name in model_names:
    # Start a tracking context for the current architecture
    with mlflow.start_run(run_name=f"Execution_{name}"):
        
        print(f"\n--- Training and Logging {name} ---")
        
        # 2. Log Hyperparameters
        mlflow.log_param("architecture", name)
        mlflow.log_param("window_size", WINDOW_SIZE)
        mlflow.log_param("features_count", len(features))
        mlflow.log_param("epochs", 20)
        
        # Build and Train the Model
        model = build_sequential_model(name)
        history = model.fit(
            X_train, y_train, 
            epochs=20, 
            batch_size=16, 
            verbose=0, 
            validation_split=0.1
        )
        
        # 3. Log Training history (Loss/Accuracy per epoch)
        for epoch, loss in enumerate(history.history['loss']):
            mlflow.log_metric("train_loss", loss, step=epoch)
        
        # 4. Evaluation Metrics
        y_probs = model.predict(X_test)
        y_preds = (y_probs > 0.5).astype(int)
        
        acc = accuracy_score(y_test, y_preds)
        f1 = f1_score(y_test, y_preds)
        rmse = np.sqrt(mean_squared_error(y_test, y_probs))
        
        mlflow.log_metric("test_accuracy", acc)
        mlflow.log_metric("test_f1", f1)
        mlflow.log_metric("test_rmse", rmse)
        
        # 5. Log the Model (FIXED: Only use 'name' to avoid Deprecation/Conflict)
        mlflow.keras.log_model(
            model, 
            name=f"MarketModel_{name}"
        )
        
        performance[name] = {'Accuracy': acc, 'F1-Score': f1, 'RMSE': rmse}
        print(f"Successfully tracked {name} in MLflow.")

# Display final summary table
print("\nFinal Results Table:")
print(pd.DataFrame(performance).T)

2026/05/03 13:32:19 INFO mlflow.tracking.fluent: Experiment with name 'RealTime_Market_Prediction' does not exist. Creating a new experiment.



--- Training and Logging RNN ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step


2026/05/03 13:32:25 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


Successfully tracked RNN in MLflow.
🏃 View run Execution_RNN at: http://127.0.0.1:5000/#/experiments/1/runs/3005884a401443c0998ce2bbc921b025
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

--- Training and Logging LSTM ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step


2026/05/03 13:32:45 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


Successfully tracked LSTM in MLflow.
🏃 View run Execution_LSTM at: http://127.0.0.1:5000/#/experiments/1/runs/bb958ff7e3734f4cb4127b0086f841d2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

--- Training and Logging GRU ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 232ms/step


2026/05/03 13:33:02 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


Successfully tracked GRU in MLflow.
🏃 View run Execution_GRU at: http://127.0.0.1:5000/#/experiments/1/runs/6e5300fc528a4c5f82a8d109ae8424ad
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

Final Results Table:
      Accuracy  F1-Score      RMSE
RNN   0.333333  0.400000  0.561260
LSTM  0.555556  0.666667  0.499564
GRU   0.611111  0.588235  0.499542
